### TOOLS

In [9]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("qwen/qwen3.6-27b", model_provider="groq")


response=model.invoke("why do parrot talks")
response

AIMessage(content='Actually, parrots don’t "talk" in the human sense—they **mimic** sounds. They are not using language to convey complex thoughts or grammar (though they can learn some context). Instead, they are capable of **vocal mimicry**, which means they can imitate a wide range of sounds, including human speech, other animals, and environmental noises.\n\nHere’s why and how they do it:\n\n### 1. **Anatomical Adaptation**\nParrots have a specialized vocal organ called the **syrinx**, located where the trachea meets the lungs. Unlike humans, who use vocal cords in the throat, parrots have two separate sets of vocal cords in the syrinx. This allows them to:\n- Produce a wide range of pitches and tones.\n- Make two sounds at once (polyphonic vocalization).\n- Mimic complex human speech patterns with remarkable accuracy.\n\n### 2. **Social Bonding**\nIn the wild, parrots are highly social animals that live in flocks. Vocalizing is a key part of maintaining group cohesion. By mimickin

### importing tools and binding tools

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
   """Get the weather at a location"""
   return f"It's sunny is {location}"

model_with_tools=model.bind_tools([get_weather])

### Calling the Tools

In [11]:
# tool call
response =model_with_tools.invoke("What's the weather like in Edinburgh?")
print(response)
for tool_call in response.tool_calls:
   # View tool calls made by the model
   print(f"Tool:{tool_call['name']}")
   print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'nj717f6pf', 'function': {'arguments': '{"location":"Edinburgh"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_time': 0.073430211, 'completion_tokens_details': None, 'prompt_time': 0.022098309, 'prompt_tokens_details': None, 'queue_time': 0.05203656, 'total_time': 0.09552852}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0fca4-0370-7fd2-8424-5f95a0fda4fa-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Edinburgh'}, 'id': 'nj717f6pf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 27, 'total_tokens': 305}
Tool:get_weather
Args: {'location': 'Edinburgh'}


### Tool Execution Loops

A tool execution loop is the process where an LLM repeatedly decides whether it needs a tool, calls the tool, receives the result, and uses that result to continue generating an answer until the task is completed.

In [ ]:

# Step 1: Model generates tool calls
messages = [{"role":"user","content":" What is the weather like in Edinburhg?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step:2 Execute Tools and collect Results
for tool_call in ai_msg.tool_calls:
   # Execute the tool with the generated arguments
   tool_result = get_weather.invoke(tool_call)
   messages.append(tool_result)

# Step 3: Pass results back to model for final response 
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in edinburgh is 72 F and sunny"

It's currently **sunny** in Edinburgh! 🌞 That's a nice break from the typical Scottish weather. Enjoy the sunshine!


: 